<a href="https://colab.research.google.com/github/twna/Machine-learning-project-to-start/blob/main/duelling_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duelling Networks

## Problem statement

An evil warlord has discovered an ancient tunnel (represented as a neural network) connected to your city's sewers. He plans to send deadly tidal waves through it to attack your city! Thankfully, you have a plan. You know there is an even older, prehistoric tunnel (also represented as a neural network) that connects to the city sewers at the point just before the warlord's waves reach the city. Let us call this point the mixing chamber.

You are going to duel him for the fate of the city by sending counter waves of your own through the prehistoric tunnel to neutralize the warlords' attacks. Aim to get the energy of the resultant waves (L2 norm of combined neural network output) to be as close to 0 as possible. The warlord will send 5 waves, and his wave patterns are known to you upfront.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
warlord_attacks = [
    [-0.2887,  2.1014, -1.1875, -1.5572, -1.0120, -1.7173,  0.4706,  3.0802, -0.2799, -2.0404,  0.3002, -2.9843],
    [0.0000,  0.0000,  0.0000, -2.9875, -4.0940,  3.0872,  0.0000,  0.0000, 0.0000,  0.0000,  0.0000,  0.0000],
    [0.000,  1.3644,  1.1336, -0.4226, -1.4847, -0.81096,  0.81096,  1.4847,  0.4226, -1.1336, -1.3644,  5.2454e-07],
    [0.8171, -0.7760, -1.2905,  0.5329,  1.1489,  0.1438,  0.8859, -0.3775, -2.2911,  0.7330, -0.9208, -0.4634],
    [0.7770, 0.9937, 0.3484, 1.2489, 1.5510, 1.3883, 1.9447, 1.9054, 2.4041, 2.2864, 2.5607, 3.6302]
]

In [ ]:
class DuellingNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        torch.manual_seed(42)

        # Warlord's ancient tunnel
        self.warlord = nn.Sequential(
            nn.Linear(12, 32), nn.LeakyReLU(0.01),
            nn.Linear(32, 64), nn.LeakyReLU(0.01),
            nn.Linear(64, 128), nn.LeakyReLU(0.01),
            nn.Linear(128, 256), nn.LeakyReLU(0.01),
            nn.Linear(256, 128),
        )

        # Your prehistoric tunnel
        self.defender = nn.Sequential(
            nn.Linear(12, 32), nn.LeakyReLU(0.01),
            nn.Linear(32, 64), nn.LeakyReLU(0.01),
            nn.Linear(64, 128), nn.LeakyReLU(0.01),
            nn.Linear(128, 256), nn.LeakyReLU(0.01),
            nn.Linear(256, 128),
        )

        # Waves meet in a mixing chamber
        self.mixing_weights = nn.Parameter(torch.randn(128) * 0.1)
        self.mixing_bias = nn.Parameter(torch.randn(128) * 0.1)

        # City gates
        self.city_gates = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, 1))

        # The tunnels are literally set in stone, they are not changing!
        for param in self.parameters():
            param.requires_grad = False

        self.eval();

    def mixing_chamber(self, warlord_force, defender_force):
        interaction = warlord_force * defender_force

        resonance = (
            self.mixing_weights * interaction
            + warlord_force
            + defender_force
            + self.mixing_bias
        )

        return resonance

    def forward(self, attack_wave, counter_wave):
        warlord_force = self.warlord(attack_wave)
        defender_force = self.defender(counter_wave)

        # The forces interact in the mixing chamber
        mixed_waves = self.mixing_chamber(warlord_force, defender_force)

        # The mixed waves crash onto the city gates!!
        destruction = self.city_gates(mixed_waves)

        return destruction

In [ ]:
def assess_defense(
    network: nn.Module,
    attacks: list[list[float]], counter_waves: list[list[float]],
) -> list[float]:
    destructions = []
    energy_limit = 100.0
    tolerance = 0.001

    with torch.no_grad():
        for i, (atk, ctr) in enumerate(zip(attacks, counter_waves)):
            atk = torch.FloatTensor(atk)
            ctr = torch.FloatTensor(ctr)

            d = network(atk, ctr).item()
            e = torch.norm(ctr).item()
            ok = abs(d) < tolerance and e <= energy_limit
            print(f"Attack {i+1}: destruction={d:+.5f} counterwave energy={e:.1f}/{energy_limit}  {'OK' if ok else 'FAIL'}")
            destructions.append(d)
    return destructions

If you dont do anything, this is what will happen to the city:

In [ ]:
# All inputs as zero
null_input = [[0] * 12] * 5

In [ ]:
duel = DuellingNetwork()
destruction = assess_defense(duel, warlord_attacks, null_input)

Attack 1: destruction=+0.10676 counterwave energy=0.0/100.0  FAIL
Attack 2: destruction=+0.10974 counterwave energy=0.0/100.0  FAIL
Attack 3: destruction=+0.11659 counterwave energy=0.0/100.0  FAIL
Attack 4: destruction=+0.11766 counterwave energy=0.0/100.0  FAIL
Attack 5: destruction=+0.11976 counterwave energy=0.0/100.0  FAIL


### Your task

Submit a list of 5 counter-wave tensors to match each of the warlord's attacks, each of shape `(12,)`. The prehistoric tunnel is fragile so all of your counter wave input cannot be too big (must have L2 norm ≤ 100.0). Please use the provided `assess_defense` function to check your score before you submit.

### Scoring

Your submission will be evaluated by the average city destruction across all warlord attacks, such that:

- Average destruction of 0.11 (close to inputing all zeros) will grant 0 points
- Average destruction below 0.001 will grant 15 points
- Average destruction between 0.001 and 0.11 will be awarded between 0 to 15 points, scaled logarithmically (i.e. score is higher as destruction approaches 0.001)

Remember that if ANY of your counter waves exceed the wave energy limit, the prehistoric tunnel will crack and you will get 0 points!

You are allowed to make 10 API submissions for this challenge.

### API submission format

In [ ]:
import requests

In [ ]:
# Copy paste your API key here from the contest webpage, don't show to others!
api_key = ""

In [ ]:
def make_payload(list_of_counterwaves):
    return {"solution": list_of_counterwaves}

In [ ]:
def post_answer(payload: dict):
    url = "https://competitions.aiolympiad.my/api/maio2026_qualis/maio2026_duelling_networks"
    response = requests.post(url=url, json=payload, headers={"X-API-Key": api_key})
    if response.status_code == 200:
        return response.json()
    else:
        return (
            f"Failed to submit, status code is {response.status_code}\n{response.text}"
        )

Example API submission:

```python
>>> null_input = [[0] * 12] * 5
>>> payload = make_payload(null_input)
>>> post_answer(payload=payload)
{'status': 'SUCCESS',
 'message': 'Answer for challenge maio2026_duelling_networks submitted successfully on 2026-02-20 00:00:00.00+00:00. Total submissions is 1 / 10.'}
```

## Your work below!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Assuming DuellingNetwork and warlord_attacks are already defined
duel = DuellingNetwork()

# Pick one attack (e.g., the first)
attack = torch.FloatTensor(warlord_attacks[0])

# Initialize counter wave as a leaf tensor
counter_wave = torch.randn(12, requires_grad=True)
counter_wave.data *= 0.1  # small initial values

# Optimizer
optimizer = optim.Adam([counter_wave], lr=0.03)

# Loss function: destruction squared + energy penalty
def loss_fn(network, atk, ctr, alpha=0):
    d = network(atk, ctr)
    energy = torch.norm(ctr)
    return d.pow(2) + alpha * energy.pow(2)

# Optimization loop
for step in range(20000):
    optimizer.zero_grad()
    loss = loss_fn(duel, attack, counter_wave)
    loss.backward()
    optimizer.step()

    # Project to satisfy energy constraint
    with torch.no_grad():
        norm = torch.norm(counter_wave)
        if norm > 100.0:
            counter_wave *= 100.0 / norm

    if step % 100 == 0:
        d_val = duel(attack, counter_wave).item()
        e_val = torch.norm(counter_wave).item()
        print(f"Step {step}, destruction={d_val:.6f}, energy={e_val:.2f}, loss={loss.item():.6f}")

# Final optimized counter wave
final_wave = counter_wave.detach().tolist()
print("Final counter wave:", final_wave)


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Assumes DuellingNetwork, warlord_attacks, and assess_defense are already defined.

# -----------------------------
# 1) Model setup (deterministic + fixed weights)
# -----------------------------
torch.manual_seed(42)
np.random.seed(42)

duel = DuellingNetwork()
duel.eval()  # IMPORTANT: disable dropout / training-mode behavior
for p in duel.parameters():
    p.requires_grad_(False)  # IMPORTANT: we optimize ctr only


# -----------------------------
# 2) Utilities
# -----------------------------
ENERGY_LIMIT = 100.0
# Safety margin to avoid float32 rounding pushing slightly > 100 on strict checks
SAFE_RADIUS = ENERGY_LIMIT * (1.0 - 1e-8)

def project_l2_ball_(x: torch.Tensor, radius: float) -> None:
    # in-place projection onto L2 ball
    n = torch.linalg.norm(x)
    if n > radius:
        x.mul_(radius / n)

def log_status(prefix: str, step: int, d_val: float, e_val: float, loss_val: float, tol: float) -> None:
    status = "OK" if (abs(d_val) < tol and e_val <= ENERGY_LIMIT) else "FAIL"
    print(f"{prefix} step {step:5d}: destruction={d_val:+.8f}, energy={e_val:.6f}, loss={loss_val:.10f} [{status}]")


# -----------------------------
# 3) Core optimizer for ONE attack
# -----------------------------
def optimize_counterwave(
    attack,
    *,
    counter_dim: int = 12,
    tol: float = 1e-3,
    steps: int = 6000,
    lr: float = 0.05,
    restarts: int = 10,
    patience: int = 1200,        # stop a restart early if not improving
    min_improve: float = 1e-6,    # improvement threshold for patience
    device: str = "cpu",
    verbose: bool = True,
):
    """
    Finds ctr for one attack by optimizing ctr (input) with projected gradient descent.
    Returns a python list length counter_dim.

    Notes:
    - ctr is optimized in float64 for stability; cast to float32 for the forward pass.
    - Projection enforces strict energy constraint.
    - Multiple restarts dramatically increase success rate on non-convex landscapes.
    """

    atk = torch.tensor(attack, dtype=torch.float32, device=device)

    best_ctr = None
    best_abs_d = float("inf")

    for r in range(restarts):
        # Different init scales across restarts improves exploration
        # (small to larger starts)
        if r % 4 == 0:
            scale = 0.001
        elif r % 4 == 1:
            scale = 0.01
        elif r % 4 == 2:
            scale = 0.1
        else:
            scale = 1.0

        ctr = (scale * torch.randn(counter_dim, dtype=torch.float64, device=device)).requires_grad_(True)
        opt = optim.Adam([ctr], lr=lr)

        # Track per-restart improvement to decide early abandon
        best_abs_d_restart = float("inf")
        last_best_step = 0

        for step in range(steps):
            opt.zero_grad(set_to_none=True)

            # Forward uses float32 (model likely trained in float32)
            d = duel(atk, ctr.to(torch.float32))
            loss = d * d  # primary objective only (no energy penalty; projection handles energy)

            loss.backward()
            opt.step()

            # Enforce energy constraint strictly after each update
            with torch.no_grad():
                project_l2_ball_(ctr, SAFE_RADIUS)

            with torch.no_grad():
                d_val = float(d.item())
                abs_d = abs(d_val)
                e_val = float(torch.linalg.norm(ctr).item())
                loss_val = float((d * d).item())

            # Update global best
            if abs_d < best_abs_d:
                best_abs_d = abs_d
                best_ctr = ctr.detach().clone()

            # Update restart best + patience
            if abs_d + min_improve < best_abs_d_restart:
                best_abs_d_restart = abs_d
                last_best_step = step

            # Logging
            if verbose and (step % 1000 == 0 or abs_d < tol):
                log_status(f"[restart {r+1}/{restarts}]", step, d_val, e_val, loss_val, tol)

            # Success early stop
            if abs_d < tol:
                return ctr.detach().cpu().tolist()

            # Restart early if stuck
            if (step - last_best_step) >= patience:
                break

    # If no restart reached tol, return the best found (still projected safely)
    with torch.no_grad():
        project_l2_ball_(best_ctr, SAFE_RADIUS)
    return best_ctr.detach().cpu().tolist()


# -----------------------------
# 4) Solve all attacks
# -----------------------------
final_counterwaves = []
for i, atk in enumerate(warlord_attacks, start=1):
    print(f"\nOptimizing counter wave for attack {i}")
    cw = optimize_counterwave(
        atk,
        counter_dim=12,   # change if your notebook uses another size
        tol=1e-3,         # change if notebook uses another threshold (e.g., 0.008)
        steps=7000,
        lr=0.06,
        restarts=12,
        patience=1500,
        verbose=True,
    )
    final_counterwaves.append(cw)

# -----------------------------
# 5) Final assessment (provided by notebook)
# -----------------------------
assess_defense(duel, warlord_attacks, final_counterwaves)

# final_counterwaves is your submission payload


Optimizing counter wave for attack 1
[restart 1/12] step     0: destruction=+0.10676783, energy=0.207582, loss=0.0113993688 [FAIL]
[restart 1/12] step  1000: destruction=+0.04561686, energy=41.634829, loss=0.0020808978 [FAIL]
[restart 1/12] step  2000: destruction=+0.02951686, energy=78.746778, loss=0.0008712451 [FAIL]
[restart 1/12] step  3000: destruction=+0.00302839, energy=99.999999, loss=0.0000091712 [FAIL]
[restart 1/12] step  3459: destruction=+0.00099807, energy=99.999999, loss=0.0000009961 [OK]

Optimizing counter wave for attack 2
[restart 1/12] step     0: destruction=+0.10973847, energy=0.209530, loss=0.0120425317 [FAIL]
[restart 1/12] step  1000: destruction=+0.04365451, energy=41.143040, loss=0.0019057161 [FAIL]
[restart 1/12] step  2000: destruction=+0.02652456, energy=69.751601, loss=0.0007035522 [FAIL]
[restart 1/12] step  3000: destruction=+0.00403433, energy=92.777528, loss=0.0000162758 [FAIL]
[restart 1/12] step  3479: destruction=+0.00099844, energy=96.438344, los

[0.0009949281811714172,
 0.0009952522814273834,
 0.000998854637145996,
 0.0009972117841243744,
 0.0009957663714885712]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Assuming DuellingNetwork, warlord_attacks, and assess_defense are already defined
duel = DuellingNetwork()
duel.eval()

def log_status(step, d_val, e_val, loss, tolerance=0.001):
    status = "OK" if abs(d_val) < tolerance and e_val <= 100 else "FAIL"
    print(f"Step {step:5d}: destruction={d_val:+.6f}, energy={e_val:.2f}, loss={loss:.6f} [{status}]")

def optimize_counterwave(attack, steps_phase1=5000, steps_phase2=5000,
                         lr=0.02, alpha=1e-6, beta=1e-3, tolerance=0.001, restarts=10):
    # Random restart for each attack
    for r in range(restarts):
        ctr = (0.1 * torch.randn(12)).requires_grad_(True)

        optimizer = optim.Adam([ctr], lr=lr)
        atk = torch.FloatTensor(attack)

        # ---- Phase 1: energy-constrained ----
        for step in range(steps_phase1):
            optimizer.zero_grad()
            d = duel(atk, ctr)
            energy = torch.norm(ctr)
            loss = d.pow(2) + alpha * energy.pow(2)
            loss.backward()
            optimizer.step()

            # Project energy
            with torch.no_grad():
                ctr *= min(1.0, 100.0 / torch.norm(ctr))

            if step % 1000 == 0:
                log_status(step, d.item(), energy.item(), loss.item(), tolerance)

            if abs(d.item()) < tolerance and energy.item() <= 100.0:
                log_status(step, d.item(), energy.item(), loss.item(), tolerance)
                return ctr.detach().tolist()

        # ---- Phase 2: energy-maximizing ----
        for step in range(steps_phase1, steps_phase1 + steps_phase2):
            optimizer.zero_grad()
            d = duel(atk, ctr)
            energy = torch.norm(ctr)
            loss = d.pow(2)
            loss.backward()
            optimizer.step()

            # Project energy
            with torch.no_grad():
                ctr *= min(1.0, 100.0 / torch.norm(ctr))
                energy_now = torch.norm(ctr).item()
                d_now = duel(atk, ctr).item()

            if step % 1000 == 0:
                log_status(step, d.item(), energy.item(), loss.item(), tolerance)

            if abs(d.item()) < tolerance and energy.item() <= 100.0:
                log_status(step, d.item(), energy.item(), loss.item(), tolerance)
                break

        return ctr.detach().tolist()

# Optimize all 5 attacks
final_counterwaves = []
for p in duel.parameters():
    p.requires_grad_(False)
for i, atk in enumerate(warlord_attacks):
    print(f"\nOptimizing counter wave for attack {i+1}")
    cw = optimize_counterwave(atk,
                          steps_phase1=5000,
                          steps_phase2=5000,
                          lr=0.02,
                          alpha=1e-6,
                          beta=0.002,   # tuned value
                          tolerance=0.001)

    final_counterwaves.append(cw)

# Assess defense
assess_defense(duel, warlord_attacks, final_counterwaves)




Optimizing counter wave for attack 1
Step     0: destruction=+0.106994, energy=0.42, loss=0.011448 [FAIL]
Step  1000: destruction=+0.050911, energy=24.04, loss=0.003170 [FAIL]
Step  2000: destruction=+0.040801, energy=31.51, loss=0.002657 [FAIL]
Step  3000: destruction=+0.040511, energy=31.58, loss=0.002638 [FAIL]
Step  4000: destruction=+0.040355, energy=31.76, loss=0.002637 [FAIL]
Step  5000: destruction=+0.040330, energy=31.81, loss=0.001626 [FAIL]
Step  6000: destruction=+0.015172, energy=66.50, loss=0.000230 [FAIL]
Step  7000: destruction=+0.005622, energy=80.86, loss=0.000032 [FAIL]
Step  8000: destruction=+0.002279, energy=87.94, loss=0.000005 [FAIL]
Step  8338: destruction=+0.000999, energy=89.69, loss=0.000001 [OK]

Optimizing counter wave for attack 2
Step     0: destruction=+0.109279, energy=0.46, loss=0.011942 [FAIL]
Step  1000: destruction=+0.054185, energy=20.57, loss=0.003359 [FAIL]
Step  2000: destruction=+0.053357, energy=21.82, loss=0.003323 [FAIL]
Step  3000: destru

[0.0009955279529094696,
 0.0009957365691661835,
 0.0009959638118743896,
 0.012934315949678421,
 0.000995635986328125]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# Assuming DuellingNetwork, warlord_attacks, and assess_defense are already defined
duel = DuellingNetwork()
duel.eval()
for p in duel.parameters():
    p.requires_grad_(False)  # freeze weights

SAFE_RADIUS = 99.9  # safety margin below 100

def log_status(step, d_val, e_val, loss, tolerance=0.001):
    status = "OK" if abs(d_val) < tolerance and e_val <= 100 else "FAIL"
    print(f"Step {step:5d}: destruction={d_val:+.6f}, energy={e_val:.2f}, loss={loss:.6f} [{status}]")

def optimize_attack4(attack, steps=12000, lr=0.03, tolerance=0.001,
                     restarts=20, patience=2000):
    atk = torch.tensor(attack, dtype=torch.float32)  # fixed once per attack
    best_wave, best_d = None, float("inf")

    for r in range(restarts):
        # varied initialization scale
        scale = 0.1 * (1.5 ** (r % 5))
        ctr = (torch.randn(12, dtype=torch.float64) * scale).requires_grad_(True)
        optimizer = optim.Adam([ctr], lr=lr)

        stuck_counter = 0
        prev_loss = math.inf

        for step in range(steps):
            optimizer.zero_grad()
            d = duel(atk, ctr.to(torch.float32))  # forward in float32
            energy = torch.norm(ctr)
            loss = d.pow(2)
            loss.backward()
            optimizer.step()

            # strict projection with safety margin
            with torch.no_grad():
                ctr *= min(1.0, SAFE_RADIUS / torch.norm(ctr))

            if step % 2000 == 0:
                log_status(step, d.item(), energy.item(), loss.item(), tolerance)

            # early success
            if abs(d.item()) < tolerance and energy.item() <= 100.0:
                print(f"Restart {r+1}: SUCCESS at step {step}")
                return ctr.detach().tolist()

            # patience early-abandon
            if abs(loss.item() - prev_loss) < 1e-8:
                stuck_counter += 1
                if stuck_counter > patience:
                    print(f"Restart {r+1}: abandoned at step {step}")
                    break
            else:
                stuck_counter = 0
            prev_loss = loss.item()

        # track best so far
        if abs(d.item()) < best_d:
            best_d, best_wave = abs(d.item()), ctr.detach().tolist()

    print(f"Best destruction after {restarts} restarts: {best_d:.6f}")
    return best_wave

# Run optimizer for attack 4 only
print("\nOptimizing counter wave for attack 4")
cw_attack4 = optimize_attack4(warlord_attacks[3], steps=12000, lr=0.03,
                              tolerance=0.001, restarts=20, patience=2000)

# Assess defense for attack 4 only
assess_defense(duel, [warlord_attacks[3]], [cw_attack4])



Optimizing counter wave for attack 4
Step     0: destruction=+0.117881, energy=0.42, loss=0.013896 [FAIL]
Step  2000: destruction=+0.029611, energy=50.50, loss=0.000877 [FAIL]
Restart 1: SUCCESS at step 3289
Attack 1: destruction=+0.00100 counterwave energy=70.9/100.0  OK


[0.0009952522814273834]

[0.0009955279529094696,
 0.0009957365691661835,
 0.0009959638118743896,
 0.0009952522814273834,
 0.000995635986328125]

In [ ]:
results = [
    0.0009955279529094696,
    0.0009957365691661835,
    0.0009959638118743896,
    0.0009952522814273834,
    0.000995635986328125
]

tolerance = 0.001
energy_limit = 100.0  # assume energies were already checked separately

for i, d in enumerate(results, start=1):
    status = "OK" if abs(d) < tolerance else "FAIL"
    print(f"Attack {i}: destruction={d:.9f}, status={status}")


Attack 1: destruction=0.000995528, status=OK
Attack 2: destruction=0.000995737, status=OK
Attack 3: destruction=0.000995964, status=OK
Attack 4: destruction=0.000995252, status=OK
Attack 5: destruction=0.000995636, status=OK
